<div style="background:#E9FFF6; color:#440404; padding:8px; border-radius: 4px; text-align: center; font-weight: 500;">IFN619 - Data Analytics for Strategic Decision Makers</div>

# IFN619 :: C1-UnstructuredAnalytics

For this session, the focus will be on analysis of unstructured text. However, the thinking required is similar to approaches to analysing images, video, sound and other unstructured data. Primarily, the analysis is based on the notion that there are useful patterns in the unstructured data which can be obtained mathematically. By converting the data to a mathematical structure, various algorithms can be applied to the structure with the aim of identifying patterns. 

In the case of the `topic modelling` approaches below, the techniques are *stochastic* - that is they include a level of randomness and mathematically identify the probability or *likelihood* that a feature might be important. Thus, they are never deterministic, repeatable or 100% accurate, and their use needs to be mediated by a more pragmatic *useful or not* approach, rather than *right or wrong*.

In [5]:
# Import the necessary libraries
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
import pandas as pd
import json
import random

### Accessing the data via The Guardian API

See the `Accessing_the_Guardian_API.ipynb` notebook file for details on getting the data. **Note:** This approach may be used for additional data for Assignment 2.

### Read in pre-saved data

To save time, we're loading in pre-saved data that was fetched using the Guardian API.

In [6]:
# Load the data - articles from The Guardian about the Winter Olympics
file_path = "data/"
file_name = "winter_olympics_articles.json"

with open(f"{file_path}{file_name}",'r', encoding='utf-8') as fp:
    articles = json.load(fp)

print(f"Loaded {len(articles)} articles from {file_name}")

Loaded 13 articles from winter_olympics_articles.json


Each dictionary entry includes the *title [date]* as `key` and the *body text* from the article as `value`.

So the values gives us a list of documents that we can analyse.

In [7]:
# Get a list of documents
documents = list(articles.values())

# View first 400 characters of the 1st document
documents[0][:400]

'When do the 2026 Winter Olympics start? The 2026 Winter Olympics officially open in the early hours of Saturday 7 February, Australian time, with the opening ceremony at Milan’s San Siro stadium. The Games run for two weeks, culminating in the closing ceremony on 23 February in Verona at the same time of 6am AEDT. Several sports with packed schedules, including curling and ice hockey, begin a coup'

### Term Count and TF/IDF as vector inputs for Topic Modelling

Recall from B3 that we created bag of words vectors using `CountVectorizer` and `TfidfVectorizer`.

In this lecture will use these vectors as inputs for our topic modelling algorithms.

All of these analyses, approach the document as a [Bag of Words](https://en.wikipedia.org/wiki/Bag-of-words_model) model. In this approach, the order of the words don't matter. A popular approach that takes into account order is [Word embedding](https://en.wikipedia.org/wiki/Word_embedding). This session does not explore word embedding.

In [8]:
# Only count terms that in maximum of 80% of documents, and a minimum of 2 documents. 
# Count a maximum of 10000 terms, and remove common english stop words
count_vectorizer = CountVectorizer(max_df=0.80,min_df=2,max_features=10000,stop_words="english")
count_dt_matrix = count_vectorizer.fit_transform(articles.values())

In [9]:
# Get the 1000 terms identified during the vectorization process
feature_names = count_vectorizer.get_feature_names_out()
feature_names

array(['000', '10', '10th', '12', '15', '16', '17', '1994', '1998', '20',
       '2002', '2006', '2010', '2022', '2024', '2025', '2026', '20th',
       '21', '23', '25', '26', '27', '28', '30', '31', '50', '53', 'able',
       'accepted', 'according', 'accused', 'achieve', 'acknowledged',
       'acl', 'action', 'actually', 'added', 'adelaide', 'advertiser',
       'advice', 'aedt', 'aerial', 'aerials', 'afternoon', 'age', 'ago',
       'ahead', 'ahmad', 'air', 'alisa', 'allegedly', 'alongside',
       'alpine', 'alps', 'america', 'american', 'amid', 'annual',
       'anthony', 'appeal', 'arms', 'arrived', 'asian', 'assessment',
       'athlete', 'athletes', 'attack', 'attempt', 'attention',
       'australians', 'awarded', 'away', 'backed', 'baff', 'base',
       'based', 'battling', 'bearer', 'beat', 'beating', 'began', 'begin',
       'beijing', 'ben', 'best', 'better', 'bid', 'big', 'biggest', 'bit',
       'blood', 'blow', 'body', 'bradbury', 'brain', 'breadth',
       'breakthrou

In [6]:
# Take a look at the vocabulary which shows the total counts for whole collection
count_vectorizer.vocabulary_

{'2026': np.int64(16),
 'start': np.int64(624),
 'open': np.int64(464),
 'early': np.int64(211),
 'hours': np.int64(321),
 'saturday': np.int64(562),
 'february': np.int64(242),
 'time': np.int64(665),
 'opening': np.int64(465),
 'ceremony': np.int64(119),
 'milan': np.int64(421),
 'run': np.int64(556),
 'weeks': np.int64(712),
 'closing': np.int64(138),
 '23': np.int64(19),
 'aedt': np.int64(41),
 'sports': np.int64(617),
 'including': np.int64(329),
 'ice': np.int64(323),
 'begin': np.int64(82),
 'couple': np.int64(167),
 'days': np.int64(182),
 'mixed': np.int64(429),
 'team': np.int64(652),
 'just': np.int64(355),
 'missed': np.int64(425),
 'qualifying': np.int64(517),
 'despite': np.int64(194),
 'ranked': np.int64(525),
 'december': np.int64(185),
 'won': np.int64(721),
 'italy': np.int64(340),
 'final': np.int64(248),
 'event': np.int64(227),
 'men': np.int64(419),
 'game': np.int64(275),
 '12': np.int64(3),
 'related': np.int64(539),
 'youngest': np.int64(727),
 'olympian': np.i

In [11]:
# Create a new dataframe with the matrix - use titles for the index and terms for the columns
count_df = pd.DataFrame(count_dt_matrix.toarray(), columns=feature_names)
count_df

,000,10,10th,12,15,16,17,1994,1998,20,...,woman,women,won,woods,work,worst,years,yellow,youngest,zealand
0,1,1,0,3,2,1,0,0,0,0,...,0,1,1,0,0,0,1,0,1,1
1,3,0,0,0,0,1,0,1,0,0,...,0,1,3,0,1,0,0,1,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,2,0,1,0,1,2,0,0,0
3,0,0,0,0,0,0,1,0,0,0,...,0,0,1,0,0,0,0,0,0,0
4,0,0,1,1,0,1,0,0,0,1,...,0,1,1,0,0,0,0,0,0,0
5,0,0,0,1,1,0,0,1,1,0,...,1,0,0,0,0,0,3,0,0,0
6,0,0,0,0,2,2,0,0,0,0,...,0,1,1,0,0,0,2,0,1,1
7,0,0,0,1,0,2,0,1,1,0,...,0,0,3,2,0,0,2,0,1,0
8,0,0,0,0,1,2,1,0,0,3,...,3,1,1,0,0,0,1,0,0,0
9,0,0,0,0,1,0,0,0,0,0,...,0,0,0,11,1,0,0,0,0,0


Keep a list of the titles of the articles to make it easy to see the headline that relates to the topics. We can always go back to the original documents if we need to.

In [13]:
titles=list(articles.keys())
titles

['Winter Olympics 2026: what you need to know if following from Australia [2026-02-02T14:00:10Z]',
 'Australia’s upward trajectory slips off course as Winter Olympics medal search goes on | Jack Snape [2026-02-12T03:36:06Z]',
 '‘Even more special’: Jakara Anthony dusts off Winter Olympics heartbreak for historic triumph [2026-02-15T01:32:16Z]',
 'A part-time job and DJ gigs helped Lara Hamilton reach the Winter Olympics. Now she wants to put Australia on the map [2026-02-17T14:00:15Z]',
 'Valentino Guseli’s unexpected big air dream ends without a medal in all-or-nothing final: ‘I left it all out there’ [2026-02-08T00:27:13Z]',
 'From Bradbury to Bright: five of Australia’s best Winter Olympic moments | Martin Pegan [2026-02-03T14:00:08Z]',
 'Australia’s youngest Winter Olympian Indra Brown: ‘I just love the feeling of flying’ | Martin Pegan [2026-02-01T14:00:34Z]',
 'How did Australia – better known for its beaches than snow – become a consistent Winter Olympics performer? | Kieran Pen

By selecting a row from the dataframe and sorting the values (counts), we can identify the top 10 terms

In [14]:
# Sample 5 random article numbers
samples = random.sample(range(0,len(count_df)),5)

# View the associated terms
for sample in samples:
    doc = count_df.iloc[sample]
    title = titles[sample]
    top_terms = dict(count_df.iloc[sample].sort_values(ascending=False).head(10))
    print(f"[{sample}] {title}")
    print("\t- Top terms:",top_terms)
    print()

[3] A part-time job and DJ gigs helped Lara Hamilton reach the Winter Olympics. Now she wants to put Australia on the map [2026-02-17T14:00:15Z]
	- Top terms: {'hamilton': np.int64(11), 'ski': np.int64(8), 'skimo': np.int64(7), 'just': np.int64(7), 'running': np.int64(7), 'says': np.int64(6), 'skiing': np.int64(5), 'mountaineering': np.int64(4), 'going': np.int64(4), 'skis': np.int64(3)}

[10] Australia’s Jakara Anthony clinches first ever dual moguls Olympics gold [2026-02-14T11:40:32Z]
	- Top terms: {'anthony': np.int64(7), 'second': np.int64(6), 'james': np.int64(4), 'moguls': np.int64(4), 'peel': np.int64(3), 'ok': np.int64(3), 'training': np.int64(3), 'won': np.int64(3), 'event': np.int64(3), 'final': np.int64(3)}

[8] ‘Put the blinders on’: how Jakara Anthony can make Winter Olympics history [2026-02-10T02:32:34Z]
	- Top terms: {'anthony': np.int64(11), 'moguls': np.int64(7), 'says': np.int64(6), 'ski': np.int64(5), 'season': np.int64(4), 'win': np.int64(4), 'field': np.int64(4),

#### Create a top10 terms dataframe

Using the index from the documents, create a dataframe that can hold the top10 terms for each document. We also include columns for our other analysis (tfidf, lda, nmf)

In [16]:
# Create a dataframe to hold top terms for each analysis type
terms_df = pd.DataFrame(index=count_df.index,columns=['title','count','tfidf','lda','nmf'])
terms_df['title'] = titles
terms_df

,title,count,tfidf,lda,nmf
0,Winter Olympics 2026: what you need to know if...,NaN,NaN,NaN,NaN
1,Australia’s upward trajectory slips off course...,NaN,NaN,NaN,NaN
2,‘Even more special’: Jakara Anthony dusts off ...,NaN,NaN,NaN,NaN
3,A part-time job and DJ gigs helped Lara Hamilt...,NaN,NaN,NaN,NaN
4,Valentino Guseli’s unexpected big air dream en...,NaN,NaN,NaN,NaN
5,From Bradbury to Bright: five of Australia’s b...,NaN,NaN,NaN,NaN
6,Australia’s youngest Winter Olympian Indra Bro...,NaN,NaN,NaN,NaN
7,How did Australia – better known for its beach...,NaN,NaN,NaN,NaN
8,‘Put the blinders on’: how Jakara Anthony can ...,NaN,NaN,NaN,NaN
9,Ice in his veins: Australian skier Cooper Wood...,NaN,NaN,NaN,NaN


Populate the count column with data created by the count vectorizer.

In [17]:
#For each doc, get the 10 columns with the largest counts
for idx in terms_df.index:
    counts = dict(count_df.loc[idx].sort_values(ascending=False).head(10))
    #print(counts)
    terms_df.at[idx,'count'] = list(counts.keys()) # Just the list of terms

terms_df

,title,count,tfidf,lda,nmf
0,Winter Olympics 2026: what you need to know if...,"[ice, 2026, new, athletes, italy, aedt, just, ...",NaN,NaN,NaN
1,Australia’s upward trajectory slips off course...,"[anthony, team, competition, final, training, ...",NaN,NaN,NaN
2,‘Even more special’: Jakara Anthony dusts off ...,"[moguls, anthony, dual, single, event, time, f...",NaN,NaN,NaN
3,A part-time job and DJ gigs helped Lara Hamilt...,"[hamilton, ski, skimo, just, running, says, sk...",NaN,NaN,NaN
4,Valentino Guseli’s unexpected big air dream en...,"[final, guseli, said, score, best, 50, air, ru...",NaN,NaN,NaN
5,From Bradbury to Bright: five of Australia’s b...,"[bradbury, event, skiing, bronze, high, added,...",NaN,NaN,NaN
6,Australia’s youngest Winter Olympian Indra Bro...,"[just, indra, time, brown, says, want, halfpip...",NaN,NaN,NaN
7,How did Australia – better known for its beach...,"[medals, moguls, sports, funding, olympian, te...",NaN,NaN,NaN
8,‘Put the blinders on’: how Jakara Anthony can ...,"[anthony, moguls, says, ski, season, win, fiel...",NaN,NaN,NaN
9,Ice in his veins: Australian skier Cooper Wood...,"[woods, said, just, cooper, lot, time, final, ...",NaN,NaN,NaN


The [TF/IDF](https://en.wikipedia.org/wiki/Tf–idf) algorithm takes the term frequencies for a document and divides them by the frequencies of the terms in the whole collection.


In [12]:
# Only count terms that in maximum of 80% of documents, and a minimum of 2 documents. 
# Count a maximum of 10000 terms, and remove common english stop words
tfidf_vectorizer = TfidfVectorizer(
    max_df=0.80, min_df=2, max_features=10000, stop_words="english"
)

In [13]:
# Get the document vectors
tfidf_dt_matrix = tfidf_vectorizer.fit_transform(articles.values())

# Display the vector for the first document
tfidf_dt_matrix.toarray()[0]

array([0.05782947, 0.05782947, 0.        , 0.13860385, 0.07100469,
       0.03854199, 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.05782947, 0.        , 0.10650704, 0.        ,
       0.        , 0.19692714, 0.        , 0.        , 0.10256164,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.05782947, 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.2313179 , 0.        , 0.        , 0.        ,
       0.        , 0.042051  , 0.09240257, 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.04620128, 0.05782947,
       0.05782947, 0.        , 0.        , 0.        , 0.03282119,
       0.        , 0.        , 0.        , 0.05782947, 0.        ,
       0.03854199, 0.17751174, 0.        , 0.05782947, 0.        ,
       0.042051  , 0.        , 0.        , 0.        , 0.     

In [14]:
# Create a dataframe for TF/IDF
tfidf_df = pd.DataFrame(tfidf_dt_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out())
#tfidf_df

In [15]:
# Update the terms dataframe with TF/IDF
for idx in terms_df.index:
    tfidf = dict(tfidf_df.loc[idx].sort_values(ascending=False).head(10))
    #print(counts)
    terms_df.at[idx,'tfidf'] = list(tfidf.keys()) 

terms_df

,title,count,tfidf,lda,nmf
0,Winter Olympics 2026: what you need to know if...,"[2026, ice, new, athletes, aedt, day, ski, wat...","[ice, aedt, watch, 2026, athletes, table, moun...",NaN,NaN
1,Australia’s upward trajectory slips off course...,"[team, anthony, final, competition, medals, tr...","[anthony, team, competition, air, snowboarder,...",NaN,NaN
2,‘Even more special’: Jakara Anthony dusts off ...,"[moguls, anthony, dual, single, event, just, f...","[moguls, anthony, dual, single, course, event,...",NaN,NaN
3,A part-time job and DJ gigs helped Lara Hamilt...,"[hamilton, ski, skimo, just, running, says, sk...","[hamilton, skimo, running, ski, mountaineering...",NaN,NaN
4,Valentino Guseli’s unexpected big air dream en...,"[final, guseli, said, time, score, air, best, ...","[guseli, final, air, 50, landing, score, said,...",NaN,NaN
5,From Bradbury to Bright: five of Australia’s b...,"[bradbury, event, high, bronze, skiing, way, s...","[bradbury, smith, high, bronze, city, event, s...",NaN,NaN
6,Australia’s youngest Winter Olympian Indra Bro...,"[just, indra, brown, time, says, want, halfpip...","[indra, want, just, brown, says, ve, time, fam...",NaN,NaN
7,How did Australia – better known for its beach...,"[medals, moguls, sports, team, funding, olympi...","[medals, moguls, funding, aerial, sports, olym...",NaN,NaN
8,‘Put the blinders on’: how Jakara Anthony can ...,"[anthony, moguls, says, ski, win, field, seaso...","[anthony, says, moguls, field, makes, woman, s...",NaN,NaN
9,Ice in his veins: Australian skier Cooper Wood...,"[woods, said, just, cooper, lot, final, win, t...","[woods, cooper, said, just, lot, perisher, nsw...",NaN,NaN


In [16]:
# Compare Counts and TF/IDF

# Sample 5 random articles
samples = random.sample(range(0,len(terms_df)),5)

for sample in samples:
    doc = terms_df.iloc[sample]
    print(f"[{sample}] {doc['title']}")
    print("\t>> Counts:\t",doc['count'])
    print("\t>> TFIDF:\t",doc['tfidf'])
    print()

[1] Australia’s upward trajectory slips off course as Winter Olympics medal search goes on | Jack Snape [2026-02-12T03:36:06Z]
	>> Counts:	 ['team', 'anthony', 'final', 'competition', 'medals', 'training', 'event', '000', 'air', 'big']
	>> TFIDF:	 ['anthony', 'team', 'competition', 'air', 'snowboarder', '000', 'compete', 'final', 'morning', 'training']

[6] Australia’s youngest Winter Olympian Indra Brown: ‘I just love the feeling of flying’ | Martin Pegan [2026-02-01T14:00:34Z]
	>> Counts:	 ['just', 'indra', 'brown', 'time', 'says', 'want', 'halfpipe', 've', 'family', 'ski']
	>> TFIDF:	 ['indra', 'want', 'just', 'brown', 'says', 've', 'time', 'family', 'trick', 'landing']

[12] Morning Mail: Taylor sets up Liberal spill, millionaires call for higher taxes, Australia’s skiing hope slips up [2026-02-11T19:40:12Z]
	>> Counts:	 ['morning', 'smith', 'shock', 'australians', 'today', 'able', 'sydney', 'set', 'action', 'week']
	>> TFIDF:	 ['morning', 'shock', 'smith', 'today', 'australians', 

### Topic modelling with Latent Dirichlet Allocation (LDA)

[LDA](https://en.wikipedia.org/wiki/Latent_Dirichlet_allocation) is an algorithm for obtaining *topics* (a list of terms) from a document-term matrix. It is a generative probabilistic approach to *decomposition* of the document-term matrix into 2 factor matrices: document-topic and topic-term.

![img](https://editor.analyticsvidhya.com/uploads/26864dtm.JPG)

*Source: [Analytics Vidhya](https://www.analyticsvidhya.com/blog/2021/06/part-2-topic-modeling-and-latent-dirichlet-allocation-lda-using-gensim-and-sklearn/)*

The LDA model requires the number of topics to be set in advance. As it is a generative model, it also runs over a number of iterations. These values usually need to be experimented with to obtain quality topics.

In [18]:
# Set number of topics
num_topics = 20
# Set max number of iteractions
max_iterations = 20

# Create the model
lda_model = LatentDirichletAllocation(n_components=num_topics,max_iter=max_iterations,learning_method='online')

# Fit the model to the data, and use the model to transform the data (do the decomposition)
doc_topic_matrix = lda_model.fit_transform(count_dt_matrix)

# Obtain the topics
topic_term_matrix = lda_model.components_

#### View the topics

In [18]:
# Get the topics and their terms
lda_topic_dict = {}
for index, topic in enumerate(topic_term_matrix):
    zipped = zip(feature_names, topic)
    top_terms=dict(sorted(zipped, key = lambda t: t[1], reverse=True)[:10])
    #print(top_terms)
    top_terms_list= {key : round(top_terms[key], 4) for key in top_terms.keys()}
    lda_topic_dict[f"topic_{index}"] = top_terms_list

# Print the topics with their terms    
for k,v in lda_topic_dict.items():
    print(k)
    print(v)
    print()

topic_0
{'guseli': np.float64(0.1477), 'final': np.float64(0.1454), 'just': np.float64(0.1449), 'team': np.float64(0.143), '2026': np.float64(0.1419), 'february': np.float64(0.1404), 'ice': np.float64(0.1397), 'skiing': np.float64(0.1373), 'table': np.float64(0.1365), 'watch': np.float64(0.1364)}

topic_1
{'skimo': np.float64(0.1415), 'just': np.float64(0.137), 'hamilton': np.float64(0.1367), 'ski': np.float64(0.1357), 'far': np.float64(0.1345), 'home': np.float64(0.134), 'mountaineering': np.float64(0.1339), 'career': np.float64(0.1316), 'going': np.float64(0.1304), 'news': np.float64(0.13)}

topic_2
{'morning': np.float64(7.5453), 'day': np.float64(4.7624), 'government': np.float64(4.7447), 'new': np.float64(3.8532), 'people': np.float64(3.8531), 'state': np.float64(3.8381), 'crossword': np.float64(3.8275), 'week': np.float64(3.8267), 'sign': np.float64(3.8262), 'follow': np.float64(3.8257)}

topic_3
{'moguls': np.float64(0.177), 'single': np.float64(0.1445), 'anthony': np.float64(0.

#### List of topics for each document

In [19]:
doc_topic_matrix

array([[1.98412698e-04, 1.98412698e-04, 1.98412700e-04, 1.98412698e-04,
        1.98412700e-04, 1.98412698e-04, 1.98412698e-04, 9.96230159e-01,
        1.98412700e-04, 1.98412698e-04, 1.98412701e-04, 1.98412698e-04,
        1.98412701e-04, 1.98412698e-04, 1.98412698e-04, 1.98412698e-04,
        1.98412698e-04, 1.98412698e-04, 1.98412698e-04, 1.98412702e-04],
       [1.71821306e-04, 1.71821306e-04, 1.71821308e-04, 1.71821306e-04,
        1.71821308e-04, 1.71821306e-04, 1.71821306e-04, 1.71821308e-04,
        1.71821308e-04, 1.71821306e-04, 1.71821351e-04, 1.71821306e-04,
        1.71821309e-04, 1.71821306e-04, 1.71821306e-04, 1.71821306e-04,
        1.71821306e-04, 1.71821306e-04, 1.71821306e-04, 9.96735395e-01],
       [2.07468880e-04, 2.07468880e-04, 2.07468882e-04, 2.07468880e-04,
        2.07468883e-04, 2.07468880e-04, 2.07468880e-04, 2.07468882e-04,
        2.07468884e-04, 2.07468880e-04, 9.96058091e-01, 2.07468880e-04,
        2.07468884e-04, 2.07468880e-04, 2.07468880e-04, 2.0746

#### Update the terms matrix

In [20]:
for idx,topic in enumerate(doc_topic_matrix):
    topic_num = topic.argmax()
    top_topic = lda_topic_dict[f"topic_{topic_num}"]
    terms_df.at[idx,'lda'] = list(top_topic.keys())

terms_df

,title,count,tfidf,lda,nmf
0,Winter Olympics 2026: what you need to know if...,"[2026, ice, new, athletes, aedt, day, ski, wat...","[ice, aedt, watch, 2026, athletes, table, moun...","[ice, 2026, athletes, new, skiing, ski, just, ...",NaN
1,Australia’s upward trajectory slips off course...,"[team, anthony, final, competition, medals, tr...","[anthony, team, competition, air, snowboarder,...","[indra, just, time, team, brown, anthony, comp...",NaN
2,‘Even more special’: Jakara Anthony dusts off ...,"[moguls, anthony, dual, single, event, just, f...","[moguls, anthony, dual, single, course, event,...","[final, anthony, moguls, event, time, said, se...",NaN
3,A part-time job and DJ gigs helped Lara Hamilt...,"[hamilton, ski, skimo, just, running, says, sk...","[hamilton, skimo, running, ski, mountaineering...","[just, hamilton, woods, ski, skiing, running, ...",NaN
4,Valentino Guseli’s unexpected big air dream en...,"[final, guseli, said, time, score, air, best, ...","[guseli, final, air, 50, landing, score, said,...","[final, anthony, moguls, event, time, said, se...",NaN
5,From Bradbury to Bright: five of Australia’s b...,"[bradbury, event, high, bronze, skiing, way, s...","[bradbury, smith, high, bronze, city, event, s...","[final, anthony, moguls, event, time, said, se...",NaN
6,Australia’s youngest Winter Olympian Indra Bro...,"[just, indra, brown, time, says, want, halfpip...","[indra, want, just, brown, says, ve, time, fam...","[indra, just, time, team, brown, anthony, comp...",NaN
7,How did Australia – better known for its beach...,"[medals, moguls, sports, team, funding, olympi...","[medals, moguls, funding, aerial, sports, olym...","[medals, moguls, sports, team, olympian, fundi...",NaN
8,‘Put the blinders on’: how Jakara Anthony can ...,"[anthony, moguls, says, ski, win, field, seaso...","[anthony, says, moguls, field, makes, woman, s...","[anthony, moguls, says, ski, win, season, fiel...",NaN
9,Ice in his veins: Australian skier Cooper Wood...,"[woods, said, just, cooper, lot, final, win, t...","[woods, cooper, said, just, lot, perisher, nsw...","[just, hamilton, woods, ski, skiing, running, ...",NaN


#### Compare approaches

In [21]:
# Sample 5 random articles
samples = random.sample(range(0,len(terms_df)),5)

for sample in samples:
    doc = terms_df.iloc[sample]
    print(f"[{sample}] {doc['title']}")
    print("\t>> Counts:\t",doc['count'])
    print("\t>> TFIDF:\t",doc['tfidf'])
    print("\t>> LDA:\t\t",doc['lda'])
    print()

[11] Morning Mail: Lake Cargelligo suspect’s past revealed, supermarket ‘per unit’ prices under scrutiny, shark attack spike [2026-02-18T20:23:22Z]
	>> Counts:	 ['morning', 'history', 'including', 'new', 'government', 'people', 'day', 'children', 'crossword', 'asian']
	>> TFIDF:	 ['morning', 'children', 'government', 'history', 'people', 'update', 'reports', 'allegedly', 'uk', 'asian']
	>> LDA:		 ['morning', 'day', 'government', 'new', 'people', 'state', 'crossword', 'week', 'sign', 'follow']

[2] ‘Even more special’: Jakara Anthony dusts off Winter Olympics heartbreak for historic triumph [2026-02-15T01:32:16Z]
	>> Counts:	 ['moguls', 'anthony', 'dual', 'single', 'event', 'just', 'final', 'time', 'course', 'lot']
	>> TFIDF:	 ['moguls', 'anthony', 'dual', 'single', 'course', 'event', 'just', 'needed', 'special', 'help']
	>> LDA:		 ['final', 'anthony', 'moguls', 'event', 'time', 'said', 'second', 'guseli', 'run', 'dual']

[9] Ice in his veins: Australian skier Cooper Woods embraces pres

### Topic modelling with Non-negative Matrix Factorisation (NMF)


[NMF](https://en.wikipedia.org/wiki/Latent_Dirichlet_allocation) is a different algorithm for obtaining *topics* (a list of terms) from a document-term matrix. It also factorises the document-term matrix into 2 factor matrices: document-topic and topic-term.

In [22]:
# Set the number of topics
num_topics = 20

# Create the model
nmf_model = NMF(n_components=num_topics,init='random',beta_loss='frobenius')

# Fit the model to the data and use it to transform the data
doc_topic_nmf = nmf_model.fit_transform(tfidf_dt_matrix)

topic_term_nmf = nmf_model.components_

In [23]:
# Get the topics and their terms
nmf_topic_dict = {}
for index, topic in enumerate(topic_term_nmf):
    zipped = zip(feature_names, topic)
    top_terms=dict(sorted(zipped, key = lambda t: t[1], reverse=True)[:10])
    #print(top_terms)
    top_terms_list= {key : round(top_terms[key], 4) for key in top_terms.keys()}
    nmf_topic_dict[f"topic_{index}"] = top_terms_list

# Print the topics with their terms    
for k,v in nmf_topic_dict.items():
    print(k)
    print(v)
    print()

topic_0
{'time': np.float64(1.5027), 'just': np.float64(1.3346), 'ski': np.float64(0.5943), 'snow': np.float64(0.571), 'related': np.float64(0.552), 'week': np.float64(0.5161), 'skiing': np.float64(0.4987), 'jakara': np.float64(0.4772), 'anthony': np.float64(0.4772), 'years': np.float64(0.4458)}

topic_1
{'guseli': np.float64(0.8604), 'final': np.float64(0.6183), 'air': np.float64(0.4303), '50': np.float64(0.3847), 'landing': np.float64(0.3414), 'score': np.float64(0.3322), 'said': np.float64(0.2657), 'round': np.float64(0.263), 'best': np.float64(0.2612), 'big': np.float64(0.2355)}

topic_2
{'bradbury': np.float64(1.3908), 'smith': np.float64(0.6954), 'high': np.float64(0.6742), 'bronze': np.float64(0.6742), 'city': np.float64(0.6166), 'event': np.float64(0.5662), 'national': np.float64(0.5556), 'speed': np.float64(0.5556), 'nation': np.float64(0.5556), 'later': np.float64(0.5556)}

topic_3
{'anthony': np.float64(0.8401), 'snowboarder': np.float64(0.653), 'air': np.float64(0.6528), 'c

#### Update the terms matrix

In [24]:
for idx,topic in enumerate(doc_topic_nmf):
    topic_num = topic.argmax()
    top_topic = nmf_topic_dict[f"topic_{topic_num}"]
    terms_df.at[idx,'nmf'] = list(top_topic.keys())

terms_df

,title,count,tfidf,lda,nmf
0,Winter Olympics 2026: what you need to know if...,"[2026, ice, new, athletes, aedt, day, ski, wat...","[ice, aedt, watch, 2026, athletes, table, moun...","[ice, 2026, athletes, new, skiing, ski, just, ...","[aedt, ice, watch, 2026, table, mountaineering..."
1,Australia’s upward trajectory slips off course...,"[team, anthony, final, competition, medals, tr...","[anthony, team, competition, air, snowboarder,...","[indra, just, time, team, brown, anthony, comp...","[anthony, snowboarder, air, compete, competiti..."
2,‘Even more special’: Jakara Anthony dusts off ...,"[moguls, anthony, dual, single, event, just, f...","[moguls, anthony, dual, single, course, event,...","[final, anthony, moguls, event, time, said, se...","[moguls, anthony, dual, single, course, event,..."
3,A part-time job and DJ gigs helped Lara Hamilt...,"[hamilton, ski, skimo, just, running, says, sk...","[hamilton, skimo, running, ski, mountaineering...","[just, hamilton, woods, ski, skiing, running, ...","[hamilton, skimo, running, ski, mountaineering..."
4,Valentino Guseli’s unexpected big air dream en...,"[final, guseli, said, time, score, air, best, ...","[guseli, final, air, 50, landing, score, said,...","[final, anthony, moguls, event, time, said, se...","[guseli, final, air, 50, landing, score, said,..."
5,From Bradbury to Bright: five of Australia’s b...,"[bradbury, event, high, bronze, skiing, way, s...","[bradbury, smith, high, bronze, city, event, s...","[final, anthony, moguls, event, time, said, se...","[bradbury, smith, high, bronze, city, event, n..."
6,Australia’s youngest Winter Olympian Indra Bro...,"[just, indra, brown, time, says, want, halfpip...","[indra, want, just, brown, says, ve, time, fam...","[indra, just, time, team, brown, anthony, comp...","[indra, want, brown, just, says, ve, family, t..."
7,How did Australia – better known for its beach...,"[medals, moguls, sports, team, funding, olympi...","[medals, moguls, funding, aerial, sports, olym...","[medals, moguls, sports, team, olympian, fundi...","[medals, moguls, sports, aerial, team, ve, fed..."
8,‘Put the blinders on’: how Jakara Anthony can ...,"[anthony, moguls, says, ski, win, field, seaso...","[anthony, says, moguls, field, makes, woman, s...","[anthony, moguls, says, ski, win, season, fiel...","[anthony, moguls, wins, 20, laffont, beijing, ..."
9,Ice in his veins: Australian skier Cooper Wood...,"[woods, said, just, cooper, lot, final, win, t...","[woods, cooper, said, just, lot, perisher, nsw...","[just, hamilton, woods, ski, skiing, running, ...","[woods, cooper, said, just, lot, nsw, man, ve,..."


### Compare approaches

In [25]:
# Sample 5 random articles
samples = random.sample(range(0,len(terms_df)),5)

for sample in samples:
    doc = terms_df.iloc[sample]
    print(f"[{sample}] {doc['title']}")
    print("\t>> Counts:\t",doc['count'])
    print("\t>> TFIDF:\t",doc['tfidf'])
    print("\t>> LDA:\t\t",doc['lda'])
    print("\t>> NMF:\t\t",doc['nmf'])
    print()

[0] Winter Olympics 2026: what you need to know if following from Australia [2026-02-02T14:00:10Z]
	>> Counts:	 ['2026', 'ice', 'new', 'athletes', 'aedt', 'day', 'ski', 'watch', 'skiing', 'time']
	>> TFIDF:	 ['ice', 'aedt', 'watch', '2026', 'athletes', 'table', 'mountaineering', 'hours', 'new', 'italy']
	>> LDA:		 ['ice', '2026', 'athletes', 'new', 'skiing', 'ski', 'just', 'watch', 'day', 'aedt']
	>> NMF:		 ['aedt', 'ice', 'watch', '2026', 'table', 'mountaineering', 'hours', 'new', 'athletes', 'italy']

[8] ‘Put the blinders on’: how Jakara Anthony can make Winter Olympics history [2026-02-10T02:32:34Z]
	>> Counts:	 ['anthony', 'moguls', 'says', 'ski', 'win', 'field', 'season', 'woman', 'skiers', 'skier']
	>> TFIDF:	 ['anthony', 'says', 'moguls', 'field', 'makes', 'woman', 'season', 'wins', '20', 'january']
	>> LDA:		 ['anthony', 'moguls', 'says', 'ski', 'win', 'season', 'field', 'second', 'jakara', '20']
	>> NMF:		 ['anthony', 'moguls', 'wins', '20', 'laffont', 'beijing', 'skiers', 'w

### How should the 'best' approach be selected?

Keep in mind:

* Count and TF/IDF are deterministic, but LDA and NMF are stochastic
* Stochastic algorithms will always need some experimentation, and will change in effectiveness based on the kind of data that is being analysed
* As with all data analytics, we need to make decisions based on the Question and the context in which the question is asked. This means also considering the stakeholders

